# Financial Text Preparation

Download, validate, clean, label, and vectorize the financial sentiment dataset for reproducible text analysis.

In [1]:
import shutil
import subprocess
import sys

uv_path = shutil.which("uv")
if uv_path is None:
    raise RuntimeError(
        "uv was not found. Install uv or select the project's configured Python environment."
    )

subprocess.check_call([
    uv_path,
    "pip",
    "install",
    "--python",
    sys.executable,
    "kagglehub",
    "pandas",
    "scikit-learn",
])
print("Notebook dependencies installed into the active kernel environment.")

Notebook dependencies installed into the active kernel environment.


In [2]:
import os
import re
from pathlib import Path

import kagglehub
import pandas as pd
from dotenv import load_dotenv
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split

project_root = Path.cwd()
if not (project_root / "pyproject.toml").exists():
    project_root = (Path.cwd() / "..").resolve()
load_dotenv(project_root / ".env")
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

In [3]:
dataset_dir = Path(kagglehub.dataset_download("sbhatti/financial-sentiment-analysis"))
csv_candidates = list(dataset_dir.rglob("*.csv"))
dataset_path = next(
    path for path in csv_candidates
    if {"Sentence", "Sentiment"}.issubset(pd.read_csv(path, nrows=1).columns)
)
phrase_df = pd.read_csv(dataset_path).dropna(subset=["Sentence", "Sentiment"]).drop_duplicates()
phrase_df["Sentiment"] = phrase_df["Sentiment"].astype(str).str.strip().str.lower()
valid_labels = {"positive", "neutral", "negative"}
if not set(phrase_df["Sentiment"]).issubset(valid_labels):
    raise ValueError("Unexpected sentiment label detected.")
print(f"Records loaded: {len(phrase_df):,}")

100%|██████████| 276k/276k [00:00<00:00, 828kB/s]

Extracting files...
Records loaded: 5,836


In [4]:
def clean_text(text: str) -> str:
    text = re.sub(r"https?://\S+|www\.\S+", " ", str(text).lower())
    text = re.sub(r"[^a-z0-9%$€£'\-\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

phrase_df["clean_text"] = phrase_df["Sentence"].map(clean_text)
phrase_df = phrase_df[phrase_df["clean_text"].str.len() > 0].reset_index(drop=True)
phrase_df["label_encoded"] = phrase_df["Sentiment"].map({"negative": -1, "neutral": 0, "positive": 1}).astype(int)
phrase_df.to_csv(processed_dir / "financial_sentiment_clean.csv", index=False)
print(f"Saved cleaned data in repository root: {processed_dir}")
print(f"Clean records: {len(phrase_df):,}")

Saved cleaned data in repository root: /data/processed
Clean records: 5,836


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    phrase_df["clean_text"],
    phrase_df["label_encoded"],
    test_size=0.20,
    random_state=42,
    stratify=phrase_df["label_encoded"],
)

bow_vectorizer = CountVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, max_features=5000)
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, max_features=5000, sublinear_tf=True)
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)
print("BoW:", X_train_bow.shape, X_test_bow.shape)
print("TF-IDF:", X_train_tfidf.shape, X_test_tfidf.shape)

BoW: (4668, 5000) (1168, 5000)
TF-IDF: (4668, 5000) (1168, 5000)
